In [38]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    confusion_matrix,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Зареждане на данните

In [3]:
from src.data_preprocessor import load_data,remove_exact_duplicates,convert_numeric_columns,drop_constant_columns


df_train, df_test = load_data(
    r"C:\Users\gabriela.aleksandrov\Desktop\intrusion_detection_system\data\raw\corrected"
)

print("Обучаващи данни:", df_train.shape)
print("Тестови данни:", df_test.shape)

print("\nОбучаващи етикети:")
print(df_train["attack_type"].unique())
print("Брой:", df_train["attack_type"].nunique())

print("\nТестови етикети:")
print(df_test["attack_type"].unique())
print("Брой:", df_test["attack_type"].nunique())
print("Обучаващи категории:")
print(df_train["attack_category"].unique())
print("Брой:", df_train["attack_category"].nunique())

print("\nТестови категории:")
print(df_test["attack_category"].unique())
print("Брой:", df_test["attack_category"].nunique())

Обучаващи данни: (494021, 43)
Тестови данни: (311029, 41)

Обучаващи етикети:
<ArrowStringArray>
[         'normal', 'buffer_overflow',      'loadmodule',            'perl',
         'neptune',           'smurf',    'guess_passwd',             'pod',
        'teardrop',       'portsweep',         'ipsweep',            'land',
       'ftp_write',            'back',            'imap',           'satan',
             'phf',            'nmap',        'multihop',     'warezmaster',
     'warezclient',             'spy',         'rootkit']
Length: 23, dtype: str
Брой: 23

Тестови етикети:
<ArrowStringArray>
[         'normal',   'snmpgetattack',           'named',           'xlock',
           'smurf',         'ipsweep',        'multihop',          'xsnoop',
        'sendmail',    'guess_passwd',           'saint', 'buffer_overflow',
       'portsweep',             'pod',         'apache2',             'phf',
        'udpstorm',     'warezmaster',            'perl',           'satan',
      

#  Анализ на данните

### Анализ типа на колоните,общ брой  липсващи и дупликати,обем данни по класове,константни колони

In [4]:
print("------------Преди обработка-------------")
print("\nТипове колони:")
print(df_train.dtypes.value_counts())

print("\nЛипсващи стойности:")
print("Train:", df_train.isna().sum().sum())
print("Test:", df_test.isna().sum().sum())

print("\nДубликати:")
print("Train:", df_train.duplicated().sum())
print("Test:", df_test.duplicated().sum())

print("\nРазпределение на класовете — train:")
print(df_train["attack_category"].value_counts())
print(df_train["attack_category"].value_counts(normalize=True).mul(100).round(2))

print("\nРазпределение на класовете — test:")
print(df_test["attack_category"].value_counts())
print(df_test["attack_category"].value_counts(normalize=True).mul(100).round(2))

print("\nКатегориални стойности:")
for column in ["protocol_type", "service", "flag"]:
    print(f"{column}:")
    print("  Train:", df_train[column].nunique())
    print("  Test:", df_test[column].nunique())

print("\nКонстантни колони:")
constant_columns = [
    column
    for column in df_train.columns
    if df_train[column].nunique(dropna=False) <= 1
]
print(constant_columns)

------------Преди обработка-------------

Типове колони:
object    38
str        5
Name: count, dtype: int64

Липсващи стойности:
Train: 0
Test: 0

Дубликати:
Train: 348435
Test: 233738

Разпределение на класовете — train:
attack_category
dos       391458
normal     97278
probe       4107
r2l         1126
u2r           52
Name: count, dtype: int64
attack_category
dos       79.24
normal    19.69
probe      0.83
r2l        0.23
u2r        0.01
Name: proportion, dtype: float64

Разпределение на класовете — test:
attack_category
dos       229853
normal     60593
r2l        16347
probe       4166
u2r           70
Name: count, dtype: int64
attack_category
dos       73.90
normal    19.48
r2l        5.26
probe      1.34
u2r        0.02
Name: proportion, dtype: float64

Категориални стойности:
protocol_type:
  Train: 3
  Test: 3
service:
  Train: 66
  Test: 65
flag:
  Train: 11
  Test: 11

Константни колони:
['num_outbound_cmds', 'is_host_login']


### Преобразуване на тип object колони в int/float.Премахване на константните колони

In [6]:
df_train=convert_numeric_columns(df_train)
df_train = drop_constant_columns(df_train)

print("------------След обработка-------------")
print("\nТипове колони:")
print(df_train.dtypes.value_counts())
print("\nКонстантни колони:")
constant_columns = [
    column
    for column in df_train.columns
    if df_train[column].nunique(dropna=False) <= 1
]
print(len(constant_columns))

------------След обработка-------------

Типове колони:
int64      21
float64    15
str         5
Name: count, dtype: int64

Константни колони:
0


 ### Анализ на дупликатите

In [8]:
def duplicate_report(df: pd.DataFrame) -> pd.DataFrame:
    report = pd.DataFrame({
        "total_rows": df.groupby("attack_category").size(),
        "duplicate_rows": (
            df[df.duplicated(keep="first")]
            .groupby("attack_category")
            .size()
        ),
        "unique_rows": (
            df.drop_duplicates()
            .groupby("attack_category")
            .size()
        ),
    }).fillna(0).astype(int)

    report["duplicate_percentage"] = (
        report["duplicate_rows"]
        / report["total_rows"]
        * 100
    ).round(2)

    return report


print("Отчет за дубликатите в обучаващия набор:")
print(duplicate_report(df_train))

print("\nОтчет за дубликатите в тестовия набор:")
print(duplicate_report(df_test))

Отчет за дубликатите в обучаващия набор:
                 total_rows  duplicate_rows  unique_rows  duplicate_percentage
attack_category                                                               
dos                  391458          336886        54572                 86.06
normal                97278            9446        87832                  9.71
probe                  4107            1976         2131                 48.11
r2l                    1126             127          999                 11.28
u2r                      52               0           52                  0.00

Отчет за дубликатите в тестовия набор:
                 total_rows  duplicate_rows  unique_rows  duplicate_percentage
attack_category                                                               
dos                  229853          206285        23568                 89.75
normal                60593           12680        47913                 20.93
probe                  4166            1484       

In [9]:
df_train_clean = remove_exact_duplicates(df_train)

print("Train преди:", df_train.shape)
print("Train след:", df_train_clean.shape)

print(
    df_train_clean["attack_category"]
    .value_counts()
)

Train преди: (494021, 41)
Train след: (145586, 41)
attack_category
normal    87832
dos       54572
probe      2131
r2l         999
u2r          52
Name: count, dtype: int64


In [11]:
print("Оставащи дубликати:", df_train_clean.duplicated().sum())
print("Липсващи стойности:", df_train_clean.isna().sum().sum())
print("\nТипове данни:")
print(df_train_clean.dtypes.value_counts())

Оставащи дубликати: 0
Липсващи стойности: 0

Типове данни:
int64      21
float64    15
str         5
Name: count, dtype: int64


In [12]:
TARGET_COLUMNS = [
    "attack_type",
    "attack_category",
]

CATEGORICAL_COLUMNS = [
    "protocol_type",
    "service",
    "flag",
]

NUMERIC_COLUMNS = [
    column
    for column in df_train_clean.columns
    if column not in TARGET_COLUMNS + CATEGORICAL_COLUMNS
]

In [13]:
print("Брой числови характеристики:", len(NUMERIC_COLUMNS))
print("Брой категориални характеристики:", len(CATEGORICAL_COLUMNS))

Брой числови характеристики: 36
Брой категориални характеристики: 3


Описателна статистика-липсващи и уникални стойности

In [14]:
numeric_summary = (
    df_train_clean[NUMERIC_COLUMNS]
    .describe()
    .T
)

numeric_summary["missing"] = (
    df_train_clean[NUMERIC_COLUMNS]
    .isna()
    .sum()
)

numeric_summary["unique_values"] = (
    df_train_clean[NUMERIC_COLUMNS]
    .nunique()
)

numeric_summary

,count,mean,std,min,25%,50%,75%,max,missing,unique_values
duration,145586.0,132.025181,1.224157e+03,0.0,0.00,0.00,0.00,58329.0,0,2495
src_bytes,145586.0,7995.699628,1.820383e+06,0.0,0.00,147.00,288.00,693375640.0,0,3300
dst_bytes,145586.0,2859.780020,6.080979e+04,0.0,0.00,105.00,1164.75,5155468.0,0,10725
land,145586.0,0.000137,1.171998e-02,0.0,0.00,0.00,0.00,1.0,0,2
wrong_fragment,145586.0,0.020201,2.393680e-01,0.0,0.00,0.00,0.00,3.0,0,3
urgent,145586.0,0.000048,1.015038e-02,0.0,0.00,0.00,0.00,3.0,0,4
hot,145586.0,0.100174,1.426798e+00,0.0,0.00,0.00,0.00,30.0,0,22
num_failed_logins,145586.0,0.000515,2.858542e-02,0.0,0.00,0.00,0.00,5.0,0,6
logged_in,145586.0,0.491490,4.999293e-01,0.0,0.00,0.00,1.00,1.0,0,2
num_compromised,145586.0,0.026177,3.311373e+00,0.0,0.00,0.00,0.00,884.0,0,23


Проверка за отрицателни стойности

In [15]:
negative_values = (
    df_train_clean[NUMERIC_COLUMNS]
    .lt(0)
    .sum()
)

negative_values = negative_values[
    negative_values > 0
]

print("Характеристики с отрицателни стойности:")
print(
    negative_values
    if not negative_values.empty
    else "Няма открити отрицателни стойности."
)

Характеристики с отрицателни стойности:
Няма открити отрицателни стойности.


Почти константни характеристики

In [16]:
near_constant_report = []

for column in NUMERIC_COLUMNS:
    value_counts = df_train_clean[column].value_counts(
        normalize=True,
        dropna=False
    )

    most_frequent_value = value_counts.index[0]
    most_frequent_share = value_counts.iloc[0] * 100

    if most_frequent_share >= 99:
        near_constant_report.append({
            "feature": column,
            "most_frequent_value": most_frequent_value,
            "percentage": round(most_frequent_share, 2),
            "unique_values": df_train_clean[column].nunique(
                dropna=False
            ),
        })

near_constant_report = pd.DataFrame(
    near_constant_report
).sort_values(
    by="percentage",
    ascending=False
)

near_constant_report

,feature,most_frequent_value,percentage,unique_values
2,urgent,0,100.00,4
0,land,0,99.99,2
6,su_attempted,0,99.99,3
3,num_failed_logins,0,99.96,6
9,num_shells,0,99.96,3
5,root_shell,0,99.96,2
8,num_file_creations,0,99.82,18
10,num_access_files,0,99.69,7
7,num_root,0,99.60,20
11,is_guest_login,0,99.53,2


In [17]:
for column in CATEGORICAL_COLUMNS:
    train_values = set(df_train_clean[column].unique())
    test_values = set(df_test[column].unique())

    print(f"\nХарактеристика: {column}")
    print(
        "Стойности само в обучаващия набор:",
        sorted(train_values - test_values)
    )
    print(
        "Стойности само в тестовия набор:",
        sorted(test_values - train_values)
    )


Характеристика: protocol_type
Стойности само в обучаващия набор: []
Стойности само в тестовия набор: []

Характеристика: service
Стойности само в обучаващия набор: ['red_i', 'urh_i']
Стойности само в тестовия набор: ['icmp']

Характеристика: flag
Стойности само в обучаващия набор: []
Стойности само в тестовия набор: []


Анализ на категорийните характеристики по тип атака,които се появяват само в train/test сетовете

In [19]:
train_services = set(df_train["service"].unique())
test_services = set(df_test["service"].unique())

train_only_services = sorted(train_services - test_services)
test_only_services = sorted(test_services - train_services)

print("Стойности на service само в обучаващия набор:")

for service in train_only_services:
    service_rows = df_train[df_train["service"] == service]

    print(f"\nservice='{service}': {len(service_rows)} записа")
    print(service_rows["attack_category"].value_counts())


print("\nСтойности на service само в тестовия набор:")

for service in test_only_services:
    service_rows = df_test[df_test["service"] == service]

    print(f"\nservice='{service}': {len(service_rows)} записа")
    print(service_rows["attack_category"].value_counts())

Стойности на service само в обучаващия набор:

service='red_i': 1 записа
attack_category
normal    1
Name: count, dtype: int64

service='urh_i': 14 записа
attack_category
normal    14
Name: count, dtype: int64

Стойности на service само в тестовия набор:

service='icmp': 2 записа
attack_category
normal    2
Name: count, dtype: int64


Установени са минимални различия между категориалните стойности в обучаващия и тестовия набор. Услугите red_i и urh_i присъстват само в обучаващите данни, а icmp — само в два тестови записа. Всички засегнати записи принадлежат към категорията normal. Поради малкия им брой не е необходимо премахване или допълнително преобразуване; при кодирането ще бъде използвана обработка на непознати категории.

# Train/test разделяне на данните

In [22]:
X = df_train_clean.drop(
    columns=["attack_type", "attack_category"]
)

y = df_train_clean["attack_category"]

X_train, X_validation, y_train, y_validation = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

X_test = df_test.drop(
    columns=["attack_type", "attack_category"]
)

y_test = df_test["attack_category"]

In [25]:
print("Размер на тренировъчната част:", X_train.shape)
print("Размер на валидационната част:", X_validation.shape)
print("Размер на тестовата част:", X_test.shape)

print("\nРазпределение в тренировъчната част:")
print(y_train.value_counts())

print("\nРазпределение във валидационната част:")
print(y_validation.value_counts())

print("\nРазпределение в тестовата част:")
print(y_test.value_counts())

Размер на тренировъчната част: (116468, 39)
Размер на валидационната част: (29118, 39)
Размер на тестовата част: (311029, 39)

Разпределение в тренировъчната част:
attack_category
normal    70265
dos       43657
probe      1705
r2l         799
u2r          42
Name: count, dtype: int64

Разпределение във валидационната част:
attack_category
normal    17567
dos       10915
probe       426
r2l         200
u2r          10
Name: count, dtype: int64

Разпределение в тестовата част:
attack_category
dos       229853
normal     60593
r2l        16347
probe       4166
u2r           70
Name: count, dtype: int64


# Обучаване на базов Logistic Regression модел

In [27]:
CATEGORICAL_COLUMNS = [
    "protocol_type",
    "service",
    "flag",
]

NUMERIC_COLUMNS = [
    column
    for column in X_train.columns
    if column not in CATEGORICAL_COLUMNS
]


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            NUMERIC_COLUMNS,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
            CATEGORICAL_COLUMNS,
        ),
    ]
)


baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

In [28]:
baseline_pipeline.fit(
    X_train,
    y_train,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](5,)","['dos','normal','probe','r2l','u2r']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](39,)","['duration','protocol_type','service',...,'dst_host_srv_serror_rate', 'dst_host_rerror_rate','dst_host_srv_rerror_rate']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,39
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-spe

In [29]:
y_validation_pred = baseline_pipeline.predict(
    X_validation
)

In [30]:
print(
    "Accuracy:",
    round(
        accuracy_score(
            y_validation,
            y_validation_pred,
        ),
        4,
    ),
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_validation,
            y_validation_pred,
            average="macro",
        ),
        4,
    ),
)

print(
    "Weighted F1:",
    round(
        f1_score(
            y_validation,
            y_validation_pred,
            average="weighted",
        ),
        4,
    ),
)

print("\nКласификационен отчет:")
print(
    classification_report(
        y_validation,
        y_validation_pred,
        digits=4,
        zero_division=0,
    )
)

Accuracy: 0.9973
Macro F1: 0.8836
Weighted F1: 0.9972

Класификационен отчет:
              precision    recall  f1-score   support

         dos     0.9990    0.9992    0.9991     10915
      normal     0.9974    0.9984    0.9979     17567
       probe     0.9903    0.9554    0.9725       426
         r2l     0.9150    0.9150    0.9150       200
         u2r     0.8000    0.4000    0.5333        10

    accuracy                         0.9973     29118
   macro avg     0.9403    0.8536    0.8836     29118
weighted avg     0.9972    0.9973    0.9972     29118



In [31]:
CLASS_ORDER = [
    "normal",
    "dos",
    "probe",
    "r2l",
    "u2r",
]

validation_confusion_matrix = confusion_matrix(
    y_validation,
    y_validation_pred,
    labels=CLASS_ORDER,
)

confusion_matrix_df = pd.DataFrame(
    validation_confusion_matrix,
    index=[
        f"Реален: {label}"
        for label in CLASS_ORDER
    ],
    columns=[
        f"Предвиден: {label}"
        for label in CLASS_ORDER
    ],
)

confusion_matrix_df

,Предвиден: normal,Предвиден: dos,Предвиден: probe,Предвиден: r2l,Предвиден: u2r
Реален: normal,17539,9,3,15,1
Реален: dos,7,10906,1,1,0
Реален: probe,18,1,407,0,0
Реален: r2l,16,1,0,183,0
Реален: u2r,5,0,0,1,4


Baseline моделът се представя много добре, но резултатът за u2r показва ефекта от класовия дисбаланс:

разпознати са само 4 от 10 u2r записа;
5 са класифицирани като normal;
1 е класифициран като r2l;
u2r recall = 0.40, което понижава Macro F1 до 0.8836.

Високата accuracy от 99.73% се определя основно от големите класове normal и dos, затова сама по себе си не е достатъчна.

Следва да обучим същия модел с class_weight="balanced", за да направим коректно сравнение.

# Обучаване на Logistic Regression  (class="balanced")

In [32]:
balanced_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

balanced_pipeline.fit(
    X_train,
    y_train,
)

y_validation_pred_balanced = balanced_pipeline.predict(
    X_validation
)

In [33]:
print(
    "Accuracy:",
    round(
        accuracy_score(
            y_validation,
            y_validation_pred_balanced,
        ),
        4,
    ),
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_validation,
            y_validation_pred_balanced,
            average="macro",
        ),
        4,
    ),
)

print(
    "Weighted F1:",
    round(
        f1_score(
            y_validation,
            y_validation_pred_balanced,
            average="weighted",
        ),
        4,
    ),
)

print("\nКласификационен отчет:")
print(
    classification_report(
        y_validation,
        y_validation_pred_balanced,
        labels=CLASS_ORDER,
        digits=4,
        zero_division=0,
    )
)

Accuracy: 0.9839
Macro F1: 0.7244
Weighted F1: 0.9868

Класификационен отчет:
              precision    recall  f1-score   support

      normal     0.9992    0.9749    0.9869     17567
         dos     0.9985    0.9990    0.9988     10915
       probe     0.8124    0.9859    0.8908       426
         r2l     0.4133    0.9650    0.5787       200
         u2r     0.0946    0.7000    0.1667        10

    accuracy                         0.9839     29118
   macro avg     0.6636    0.9250    0.7244     29118
weighted avg     0.9919    0.9839    0.9868     29118



In [34]:
balanced_confusion_matrix = confusion_matrix(
    y_validation,
    y_validation_pred_balanced,
    labels=CLASS_ORDER,
)

balanced_confusion_matrix_df = pd.DataFrame(
    balanced_confusion_matrix,
    index=[
        f"Реален: {label}"
        for label in CLASS_ORDER
    ],
    columns=[
        f"Предвиден: {label}"
        for label in CLASS_ORDER
    ],
)

balanced_confusion_matrix_df

,Предвиден: normal,Предвиден: dos,Предвиден: probe,Предвиден: r2l,Предвиден: u2r
Реален: normal,17126,16,93,271,61
Реален: dos,7,10904,4,0,0
Реален: probe,5,0,420,0,1
Реален: r2l,2,0,0,193,5
Реален: u2r,0,0,0,3,7


Балансирният модел увеличава разпознаването на редките класове, но създава твърде много фалшиви аларми.При u2r моделът открива 7 от 10 атаки вместо 4, но прогнозира общо 74 записа като u2r, от които само 7 са правилни. Това означава 67 фалшиви положителни резултата.

При r2l:

открива 193 от 200 атаки;
но 271 нормални записа са погрешно определени като r2l;
precision пада от 0.9150 на 0.4133.

Следователно логистичната регресия без балансиране е по-добрият от двата варианта според основната метрика Macro F1.Вариантът с class_weight="balanced" показва, че пълното автоматично балансиране е прекалено агресивно.

# Random Forest

In [ ]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            NUMERIC_COLUMNS,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
            CATEGORICAL_COLUMNS,
        ),
    ]
)


random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)


random_forest_pipeline.fit(
    X_train,
    y_train,
)

y_validation_pred_rf = random_forest_pipeline.predict(
    X_validation
)

In [36]:
print(
    "Accuracy:",
    round(
        accuracy_score(
            y_validation,
            y_validation_pred_rf,
        ),
        4,
    ),
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_validation,
            y_validation_pred_rf,
            average="macro",
        ),
        4,
    ),
)

print(
    "Weighted F1:",
    round(
        f1_score(
            y_validation,
            y_validation_pred_rf,
            average="weighted",
        ),
        4,
    ),
)

print("\nКласификационен отчет:")
print(
    classification_report(
        y_validation,
        y_validation_pred_rf,
        labels=CLASS_ORDER,
        digits=4,
        zero_division=0,
    )
)

Accuracy: 0.9995
Macro F1: 0.9602
Weighted F1: 0.9995

Класификационен отчет:
              precision    recall  f1-score   support

      normal     0.9993    0.9999    0.9996     17567
         dos     1.0000    1.0000    1.0000     10915
       probe     0.9976    0.9883    0.9929       426
         r2l     0.9949    0.9750    0.9848       200
         u2r     1.0000    0.7000    0.8235        10

    accuracy                         0.9995     29118
   macro avg     0.9984    0.9326    0.9602     29118
weighted avg     0.9995    0.9995    0.9995     29118



In [37]:
random_forest_confusion_matrix = confusion_matrix(
    y_validation,
    y_validation_pred_rf,
    labels=CLASS_ORDER,
)

random_forest_confusion_matrix_df = pd.DataFrame(
    random_forest_confusion_matrix,
    index=[
        f"Реален: {label}"
        for label in CLASS_ORDER
    ],
    columns=[
        f"Предвиден: {label}"
        for label in CLASS_ORDER
    ],
)

random_forest_confusion_matrix_df

,Предвиден: normal,Предвиден: dos,Предвиден: probe,Предвиден: r2l,Предвиден: u2r
Реален: normal,17565,0,1,1,0
Реален: dos,0,10915,0,0,0
Реален: probe,5,0,421,0,0
Реален: r2l,5,0,0,195,0
Реален: u2r,3,0,0,0,7


По класове Random Forest постига:

dos: всички 10 915 записа са разпознати правилно;
normal: 17 565 от 17 567;
probe: 421 от 426;
r2l: 195 от 200;
u2r: 7 от 10.

Особено важно е, че при u2r:

recall се увеличава от 0.40 на 0.70;
precision е 1.00;
няма нормални записи, погрешно класифицирани като u2r;
останалите 3 u2r записа са определени като normal.

# DecisionTreeClassifier

In [39]:
decision_tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(
                random_state=42,
            ),
        ),
    ]
)


decision_tree_pipeline.fit(
    X_train,
    y_train,
)

y_validation_pred_dt = decision_tree_pipeline.predict(
    X_validation
)

In [40]:
print(
    "Accuracy:",
    round(
        accuracy_score(
            y_validation,
            y_validation_pred_dt,
        ),
        4,
    ),
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_validation,
            y_validation_pred_dt,
            average="macro",
        ),
        4,
    ),
)

print(
    "Weighted F1:",
    round(
        f1_score(
            y_validation,
            y_validation_pred_dt,
            average="weighted",
        ),
        4,
    ),
)

print("\nКласификационен отчет:")
print(
    classification_report(
        y_validation,
        y_validation_pred_dt,
        labels=CLASS_ORDER,
        digits=4,
        zero_division=0,
    )
)

Accuracy: 0.9993
Macro F1: 0.9033
Weighted F1: 0.9993

Класификационен отчет:
              precision    recall  f1-score   support

      normal     0.9997    0.9993    0.9995     17567
         dos     1.0000    1.0000    1.0000     10915
       probe     0.9907    0.9977    0.9942       426
         r2l     0.9751    0.9800    0.9776       200
         u2r     0.5000    0.6000    0.5455        10

    accuracy                         0.9993     29118
   macro avg     0.8931    0.9154    0.9033     29118
weighted avg     0.9993    0.9993    0.9993     29118



In [41]:
decision_tree_confusion_matrix = confusion_matrix(
    y_validation,
    y_validation_pred_dt,
    labels=CLASS_ORDER,
)

decision_tree_confusion_matrix_df = pd.DataFrame(
    decision_tree_confusion_matrix,
    index=[
        f"Реален: {label}"
        for label in CLASS_ORDER
    ],
    columns=[
        f"Предвиден: {label}"
        for label in CLASS_ORDER
    ],
)

decision_tree_confusion_matrix_df

,Предвиден: normal,Предвиден: dos,Предвиден: probe,Предвиден: r2l,Предвиден: u2r
Реален: normal,17555,0,4,2,6
Реален: dos,0,10915,0,0,0
Реален: probe,1,0,425,0,0
Реален: r2l,4,0,0,196,0
Реален: u2r,1,0,0,3,6


Decision Tree разпознава 6 от 10 u2r атаки, но генерира и 6 фалшиви u2r предупреждения от нормални записи. Затова precision е само 0.50. Random Forest разпознава 7 от 10 без нито една фалшива u2r прогноза.

При останалите класове Decision Tree е много силен:

всички dos записи са правилно разпознати;
425 от 426 probe;
196 от 200 r2l;
17 555 от 17 567 normal.

Random Forest остава избраният модел според основната метрика Macro F1.

In [44]:
print(
    "Дълбочина на Decision Tree:",
    decision_tree_pipeline
    .named_steps["classifier"]
    .get_depth(),
)

print(
    "Брой листа:",
    decision_tree_pipeline
    .named_steps["classifier"]
    .get_n_leaves(),
)

Дълбочина на Decision Tree: 29
Брой листа: 163


Сложен модел с потенциал за преобучение, макар че резултатът върху validation е висок. Random Forest намалява този риск чрез комбиниране на множество различни дървета и затова постига по-добър Macro F1.